# Load Test Analysis

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
with open("./reports/artillery.meta.json") as file:
    artillery_meta = json.load(file)

with open("./reports/artillery.report.json") as file:
    artillery_data = json.load(file)

In [ ]:
print("Target:", artillery_meta["target"])
print("Timestamp:", artillery_meta["timestamp"])

## Overview

### Request Summary

This chart shows a high-level overview of the total number of requests executed during the load test and their outcomes.

The bars represent three key metrics:

| Metric     | Description                                                               |
| ---------- | ------------------------------------------------------------------------- |
| Requests   | Total number of HTTP requests sent by the load test                       |
| Responses  | Number of requests that successfully received a response from the server  |
| Failed     | Number of virtual users or requests that failed during execution          |

This visualization helps quickly assess the **reliability of the system under load**. In an ideal scenario, the number of responses should match the number of requests, indicating that all requests were successfully handled.

Any difference between requests and responses represents **failed requests**, which may be caused by issues such as connection failures, server crashes, resource exhaustion, or network interruptions.

A low failure rate generally indicates stable behavior under the tested load, while increasing failure counts may signal that the system is approaching its operational limits.

In [ ]:
rates = artillery_data["aggregate"]["rates"]
counters = artillery_data["aggregate"]["counters"]

rps = rates["http.request_rate"]
requests = counters["http.requests"]
responses = counters["http.responses"]
failed = counters.get("vusers.failed", 0)

fig, ax = plt.subplots(figsize=(8, 5))

labels = ["Requests", "Responses", "Failed"]
values = [requests, responses, failed]

ax.bar(labels, values)

ax.set_title("Request Summary")
ax.set_ylabel("Count")

for i, v in enumerate(values):
    ax.text(i, v + (max(values) * 0.01), str(v), ha="center")

plt.tight_layout()
plt.show()

### Error Type Distribution

This chart shows the distribution of request failures by error type during the load test.

Each segment of the pie represents a specific error category and its proportion relative to all observed errors. The chart provides a quick overview of **which failure modes dominate during the test**.

The legend lists each error type together with the absolute number of occurrences. This ensures that even very small segments remain readable and that both **relative proportions and absolute counts** can be interpreted correctly.

A healthy system under the tested load should typically show **few or no errors**, while a growing share of specific error types may indicate system limits, configuration issues, or backend failures.

In [ ]:
counters = artillery_data["aggregate"]["counters"]

errors = {
    k.replace("errors.", ""): v
    for k, v in counters.items()
    if k.startswith("errors.")
}

if not errors or len(errors) == 0:
    print("No errors recorded during the load test.")
else:
    labels = list(errors.keys())
    values = list(errors.values())

    fig, ax = plt.subplots(figsize=(6, 6))

    wedges, texts, autotexts = ax.pie(
        values,
        autopct="%1.1f%%",
        startangle=90
    )

    legend_labels = [f"{k} ({v})" for k, v in errors.items()]

    ax.legend(
        wedges,
        legend_labels,
        title="Error Types",
        loc="center left",
        bbox_to_anchor=(1, 0.5)
    )

    ax.set_title("Error Type Distribution")

    plt.tight_layout()
    plt.show()

### Latency Percentiles

This plot shows the distribution of HTTP response times across several latency percentiles.

Each percentile represents the maximum response time within which a certain percentage of all requests completed:

| Percentile | Meaning                                                               |
| ---------- | --------------------------------------------------------------------- |
| p50        | 50% of all requests completed faster than this value (median latency) |
| p75        | 75% of requests completed faster than this value                      |
| p90        | 90% of requests completed faster than this value                      |
| p95        | 95% of requests completed faster than this value                      |
| p99        | 99% of requests completed faster than this value                      |
| p999       | 99.9% of requests completed faster than this value                    |

The x-axis represents the percentile level, while the y-axis shows the corresponding response time in milliseconds.

This visualization helps identify **tail latency**, which refers to the slowest requests in the system. Even when average latency is low, high percentiles such as p95 or p99 may reveal performance degradation affecting a small portion of requests.

In performance testing, percentiles are generally more informative than averages because they highlight outliers and worst-case response times. Stable systems typically show a relatively flat curve where higher percentiles do not increase sharply.

In [ ]:
lat = artillery_data["aggregate"]["summaries"]["http.response_time"]

percentiles = ["p50", "p75", "p90", "p95", "p99", "p999"]
values = [lat[p] for p in percentiles]

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(percentiles, values, marker="o", linewidth=2)

ax.set_title("HTTP Response Time Percentiles", fontsize=14)
ax.set_xlabel("Percentile")
ax.set_ylabel("Latency (ms)")

ax.grid(True, linestyle="--", alpha=0.6)

for x, y in zip(percentiles, values):
    ax.annotate(f"{y:.1f} ms", (x, y), textcoords="offset points", xytext=(0, 6), ha="center")

plt.tight_layout()
plt.show()